In [ ]:
# Run all_comparison.py to get the results printed to the console.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

hotpot_qa_median_length = 40
loogle_median_length = 471
longbench_median_length = 484

base_dir = Path("results_v9_lfm_2")
ks = [1, 2, 3, 4, 5]

datasets = [
    {
        "label": "HotpotQA",
        "file": base_dir / "hotpot_qa_ranking_analysis_v9.json",
        "median_length": hotpot_qa_median_length,
    },
    {
        "label": "Loogle (sh.)",
        "file": base_dir / "loogle_short_ranking_analysis_v9.json",
        "median_length": loogle_median_length,
    },
    {
        "label": "Loogle (l.)",
        "file": base_dir / "loogle_long_ranking_analysis_v9.json",
        "median_length": loogle_median_length,
    },
    {
        "label": "Longbench",
        "file": base_dir / "longbench_ranking_analysis_v9.json",
        "median_length": longbench_median_length,
    },
]

method_specs = [
    ("TreeFinder", "TreeFinder", "lightblue"),
    ("ContextCite", "ContextCite", "lightcoral"),
    ("TracLLM", "TracLLM", "plum"),
    ("SelfCite", "SelfCitation", "khaki"),
]

positions_by_dataset = {}
for ds in datasets:
    try:
        with ds["file"].open("r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        positions_by_dataset[ds["label"]] = {
            method_name: [[0.0] * 10 for _ in ks]
            for _, method_name, _ in method_specs
        }
        continue

    method_positions = {}
    for _, method_name, _ in method_specs:
        method_positions[method_name] = [
            data[f"k{k}_{method_name}_positions"] for k in ks
        ]
    positions_by_dataset[ds["label"]] = method_positions

# Create 4 subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
axes = axes.flatten()

# Offsets to show four methods side-by-side for each k.
offsets = [-0.27, -0.09, 0.09, 0.27]
violin_width = 0.16

for idx, ds in enumerate(datasets):
    ax = axes[idx]
    ds_label = ds["label"]
    method_data = positions_by_dataset[ds_label]

    for method_idx, (legend_name, method_name, color) in enumerate(method_specs):
        pos = np.array(ks, dtype=float) + offsets[method_idx]
        values = method_data[method_name]

        vp = ax.violinplot(
            values,
            positions=pos,
            widths=violin_width,
            showmeans=False,
            showmedians=True,
            showextrema=False,
        )

        for body in vp["bodies"]:
            body.set_facecolor(color)
            body.set_edgecolor("black")
            body.set_alpha(0.75)

        vp["cmedians"].set_color("black")
        vp["cmedians"].set_linewidth(1.2)

        # Add invisible handles once per subplot for a clean legend.
        ax.plot([], [], color=color, linewidth=8, alpha=0.75, label=legend_name)

    all_values = [
        v
        for method_name in method_data
        for per_k_values in method_data[method_name]
        for v in per_k_values
    ]
    max_rank = max(all_values) if all_values else 1.0
    y_max = max(1.0, max_rank * 1.15)

    ax.set_ylim(0, y_max)
    ax.set_xlim(0.5, 5.5)
    ax.set_xticks(ks)
    ax.set_xticklabels([str(k) for k in ks], fontsize=14)

    ax.text(
        0.98,
        0.98,
        f"Median Length of dataset: {ds['median_length']}",
        transform=ax.transAxes,
        fontsize=14,
        verticalalignment="top",
        horizontalalignment="right",
        bbox=dict(boxstyle="round", facecolor="red", alpha=0.3),
    )

    ax.set_xlabel("k", fontsize=18)
    ax.set_ylabel("Method Assigned GT Position", fontsize=18)
    ax.set_title(ds_label, fontsize=22, fontweight="bold")
    ax.legend(fontsize=12, loc="upper left")
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# Save as PDF
fig.savefig(base_dir / "ranking_comparison_all.pdf")